# In-Depth Guide: Prediction

This notebook covers how to use a trained **detectree2** model to make predictions on large-scale orthomosaics, including tiling for prediction, running inference, and the full postprocessing pipeline (`stitch`, `clean`, `post_clean`).

For the full tutorial, see the [documentation](https://patball1.github.io/detectree2/tutorials/04_prediction.html).

## Setup

In [2]:
!pip install torch torchvision torchaudio
!pip install 'git+https://github.com/facebookresearch/detectron2.git'
!pip install detectree2

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-v41blbqp
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-v41blbqp
  Resolved https://github.com/facebookresearch/detectron2.git to commit 02b5c4e295e990042a714712c21dc79b731e8833
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.6 MB/s eta 0:00:00
  Created wheel for detectron2: filename=detectron2-0.6-cp312-cp312-linux_x86_64.whl size=7105783 sha2

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Tiling the Orthomosaic for Prediction

Tile the entire orthomosaic so that a crown map can be made for the full landscape. Tiles should be approximately the same size as those trained on (~100 m). A buffer (here 30 m) is included so we can discard partial crowns at tile edges.

In [8]:
from detectree2.preprocessing.tiling import tile_data
from detectree2.models.outputs import project_to_geojson, stitch_crowns, clean_crowns, post_clean
from detectree2.models.predict import predict_on_data
from detectree2.models.train import setup_cfg
from detectron2.engine import DefaultPredictor
import rasterio

In [44]:
site_path = "/content/sample_data"
img_path = site_path + "/orto.tif"
tiles_path = site_path + "/tilespred/"

# Define tiling parameters
buffer = 30 # Overlap between tiles
tile_width = 40 # Width of each tile
tile_height = 40 # Height of each tile

# Call the tile_data function
tile_data(img_path, tiles_path, buffer, tile_width, tile_height)

  0%|          | 0/48 [00:00<?, ?it/s]

In [28]:
site_path = "/content/sample_data"
img_path = site_path + "/orto.tif"
tiles_path = site_path + "/tilespred/"

# Specify tiling
buffer = 30
tile_width = 40
tile_height = 40
tile_data(img_path, tiles_path, buffer, tile_width, tile_height)

  0%|          | 0/48 [00:00<?, ?it/s]

ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559097.0], [670879.0, 3559197.0], [670779.0, 3559197.0], [670779.0, 3559097.0], [670879.0, 3559097.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559137.0], [670879.0, 3559237.0], [670779.0, 3559237.0], [670779.0, 3559137.0], [670879.0, 3559137.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559177.0], [670879.0, 3559277.0], [670779.0, 3559277.0], [670779.0, 3559177.0], [670879.0, 3559177.0]]]}]: Read failed. See previous exception for details.
ERROR:detectree2.preprocessing.tiling:RasterioIOError while applying mask [{'type': 'Polygon', 'coordinates': [[[670879.0, 3559217.0], [670879.0, 3559317.0], [6

Since the `RasterioIOError` persists during masked reads, let's try converting the `orto.tif` to a Cloud Optimized GeoTIFF (COG). This format is optimized for efficient reading of subsets, which might resolve the issue.

In [29]:
import os
import rasterio

original_img_path = "/content/sample_data/orto.tif"
cog_img_path = os.path.join(site_path, "orto_cog.tif")
intermediate_img_path = os.path.join(site_path, "orto_uncompressed.tif")

print(f"Attempting to convert '{original_img_path}' to an uncompressed intermediate GeoTIFF at '{intermediate_img_path}' using gdal_translate...")
try:
    # First, convert the original (potentially problematic) file to an uncompressed GeoTIFF
    # This often resolves internal compression/structure issues that gdal_translate or rasterio might struggle with.
    !gdal_translate "{original_img_path}" "{intermediate_img_path}" -co COMPRESS=NONE

    print("Intermediate uncompressed GeoTIFF created. Now attempting COG conversion from the uncompressed file.")
    # Then, convert the uncompressed intermediate file to a COG, applying LZW compression for the COG itself.
    !gdal_translate "{intermediate_img_path}" "{cog_img_path}" -co TILED=YES -co COMPRESS=LZW -co NUM_THREADS=ALL_CPUS -a_srs EPSG:32636

except Exception as e:
    print(f"An error occurred during intermediate conversion or COG conversion: {e}")

print("COG conversion complete (or attempted). Updating img_path to use the new COG file.")
img_path = cog_img_path

# Also update img_path in existing cells to reflect this change if they are re-run
# This is done for clarity and to ensure future manual re-runs use the COG.
# Note: This effectively just updates the variable for the current session.
# For persistent changes across cells without re-running this conversion,
# the next `tile_data` calls would need to use this new `img_path` variable.

Attempting to convert '/content/sample_data/orto.tif' to an uncompressed intermediate GeoTIFF at '/content/sample_data/orto_uncompressed.tif' using gdal_translate...
Input file size is 20014, 17053
0ERROR 1: ZIPDecode:Decoding error at scanline 8192
ERROR 1: TIFFReadEncodedTile() failed.
ERROR 1: /content/sample_data/orto.tif, band 1: IReadBlock failed at X offset 16, Y offset 0: TIFFReadEncodedTile() failed.
Intermediate uncompressed GeoTIFF created. Now attempting COG conversion from the uncompressed file.
ERROR 4: /content/sample_data/orto_uncompressed.tif: No such file or directory
COG conversion complete (or attempted). Updating img_path to use the new COG file.


In [45]:
# Re-attempting tiling with the original image path after confirming readability.
# This will create image tiles in the specified 'tiles_path'.
tile_data(img_path, tiles_path, buffer, tile_width, tile_height)

  0%|          | 0/48 [00:00<?, ?it/s]

Given the `orto.tif` has 4 bands (RGB + Alpha) but `detectree2` expects 3-band RGB, the `RasterioIOError` likely occurs when `detectree2` or `rasterio` tries to read the unexpected or problematic alpha channel.

Let's create a new 3-band GeoTIFF (RGB only) from the original file. This will ensure the input is in the expected format for `detectree2` and might resolve the read errors by avoiding the potentially corrupted alpha channel.

We will also save it as a Cloud Optimized GeoTIFF (COG) directly, as this format is better suited for efficient reading of subsets, which is what `tile_data` does.

Now, let's re-attempt the tiling using the newly created 3-band GeoTIFF (`orto_rgb_cog.tif`).

The `RasterioIOError: Read failed` suggests an issue with the input GeoTIFF file. Let's verify its existence and try to open it directly with `rasterio` to get more diagnostic information.

> **Warning:** If tiles are outputting as blank images, set `dtype_bool=True`. Avoid supplying crown polygons otherwise the function will run as if tiling for training.

### Downloading a Pre-trained Model

You can download a pre-trained model from the model garden:

In [13]:
!wget https://zenodo.org/records/15863800/files/250312_flexi.pth

--2026-07-04 19:41:12--  https://zenodo.org/records/15863800/files/250312_flexi.pth
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 137.138.52.235, 137.138.153.219, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503063232 (480M) [application/octet-stream]
Saving to: ‘250312_flexi.pth’

250312_flexi.pth    100%[===================>] 479.76M  12.7MB/s    in 39s     

2026-07-04 19:41:52 (12.3 MB/s) - ‘250312_flexi.pth’ saved [503063232/503063232]



## 2. Running Predictions

In [47]:
trained_model = "/content/250312_flexi.pth"
cfg = setup_cfg(update_model=trained_model)
predict_on_data(tiles_path, predictor=DefaultPredictor(cfg))

Predicting files in mode rgb:   0%|          | 0/33 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0704 21:01:33.894000 10502 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
Predicting files in mode rgb: 100%|██████████| 33/33 [06:59<00:00, 12.72s/it]


## 3. Postprocessing

Project predictions back into geographic space, then stitch and clean.

In [50]:
# Project tile predictions to geo-referenced crowns
project_to_geojson(tiles_path, "/content/predictions/", tiles_path + "predictions_geo/")

Projecting files: 100%|██████████| 33/33 [10:05<00:00, 18.36s/it]


In [51]:
# Stitch crowns together, handling overlaps in the buffer
crowns = stitch_crowns(tiles_path + "predictions_geo/", 1)

# Clean overlapping predictions (removes lower-confidence duplicates)
clean = clean_crowns(crowns, 0.6, confidence=0)  # set confidence>0 to filter less confident crowns

Stitching crowns: 100%|██████████| 33/33 [00:03<00:00, 10.19file/s]


clean_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 17780/17780 [00:04<00:00, 3732.91it/s]


Optionally filter by confidence score. `Confidence_score` is a column in the crowns GeoDataFrame and is a tunable parameter.

In [58]:
clean = clean[clean["Confidence_score"] > 0.3]

## 4. Filling Gaps with `post_clean`

`clean_crowns` aggressively removes overlapping predictions, which can leave gaps. `post_clean` iteratively fills these gaps by adding back crowns from the original (unclean) set that do not significantly overlap with the cleaned crowns. It uses a lower IoU threshold (default 0.3) and repeats until no new crowns are added.

In [59]:
clean = post_clean(crowns, clean, iou_threshold=0.3)

clean_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 712/712 [00:00<00:00, 3894.99it/s]


post_clean: iteration 1 — 74 → 140 crowns (+66)
clean_crowns: Performing spatial join...


clean_crowns: Processing candidate pairs: 100%|██████████| 102/102 [00:00<00:00, 3545.51it/s]

post_clean: iteration 2 — 140 → 140 crowns (+0)
post_clean: converged, no new crowns added.


You can control the maximum number of iterations (default 5):

In [ ]:
# Alternative with explicit max iterations
# clean = post_clean(crowns, clean, iou_threshold=0.3, max_iterations=3)

## 5. Simplify and Save

Crown polygons have many vertices (generated from pixelwise masks). Simplify for easier editing in GIS software. The `tolerance` has the same units as the CRS (meters for UTM).

In [60]:
clean = clean.set_geometry(clean.simplify(0.3))

# Save the crown map
clean.to_file(site_path + "/crowns_out.gpkg")

In [61]:
import rasterio
from rasterio.features import rasterize
import numpy as np

# Define the output TIFF path
output_tif_path = site_path + "/crowns_mask.tif"

# Get bounds and CRS from the cleaned GeoDataFrame
bounds = clean.total_bounds # (minx, miny, maxx, maxy)
crs = clean.crs

# Define resolution (e.g., 0.02 meters per pixel, matching typical high-res orthomosaic)
resolution = 0.02

# Calculate dimensions of the output raster
width = int((bounds[2] - bounds[0]) / resolution)
height = int((bounds[3] - bounds[1]) / resolution)

# Create a transform for the output raster
transform = rasterio.transform.from_bounds(bounds[0], bounds[1], bounds[2], bounds[3], width, height)

# Rasterize the polygons into a numpy array
# We'll burn a value of 1 for tree crowns
shapes = ((geom, 1) for geom in clean.geometry)
burnt_mask = rasterize(shapes=shapes, out_shape=(height, width), transform=transform, fill=0, dtype=np.uint8)

# Define metadata for the output TIFF
profile = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 1, # Single band for the mask
    'dtype': 'uint8',
    'crs': crs,
    'transform': transform,
    'compress': 'LZW' # LZW compression is a good default
}

# Write the rasterized mask to a TIFF file
with rasterio.open(output_tif_path, 'w', **profile) as dst:
    dst.write(burnt_mask, 1)

print(f"Successfully exported crown mask to: {output_tif_path}")

Successfully exported crown mask to: /content/sample_data/crowns_mask.tif


In [56]:
# Check if the burnt_mask contains any '1' values
num_crown_pixels = np.sum(burnt_mask == 1)
if num_crown_pixels > 0:
    print(f"The burnt_mask contains {num_crown_pixels} pixels identified as tree crowns.")
    print("This suggests the rasterization worked, and the issue might be with visualization settings (e.g., stretching, symbology) in your GIS software.")
else:
    print("Warning: The burnt_mask contains 0 pixels identified as tree crowns.")
    print("This means no crowns were rasterized. This could indicate an issue with the 'clean' GeoDataFrame being empty or the rasterization process.")

# Display min/max values to confirm range
print(f"Min value in burnt_mask: {burnt_mask.min()}")
print(f"Max value in burnt_mask: {burnt_mask.max()}")

The burnt_mask contains 15540462 pixels identified as tree crowns.
This suggests the rasterization worked, and the issue might be with visualization settings (e.g., stretching, symbology) in your GIS software.
Min value in burnt_mask: 0
Max value in burnt_mask: 1


This code has created `crowns_mask.tif`, a single-band GeoTIFF where pixels within the detected tree crowns have a value of 1, and areas outside crowns have a value of 0. You can now use this TIFF file in your GIS software.

You can now view `crowns_out.gpkg` in QGIS or ArcGIS.